In [46]:
import pandas as pd
import numpy as np

In [47]:
file_path = "../data/processed/online_retail_II_cleaned.csv"

df = pd.read_csv(file_path)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Dataset shape:", df.shape)
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

df.head()

Dataset shape: (776844, 8)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [52]:
# Calculate the total value of each transaction line

df["TotalPrice"] = df["Quantity"] * df["Price"]

print(df[["Quantity", "Price", "TotalPrice"]].head())

print("\nNegative TotalPrice:", (df["TotalPrice"] < 0).sum())
print("Zero TotalPrice:", (df["TotalPrice"] == 0).sum())

   Quantity  Price  TotalPrice
0        12   6.95        83.4
1        12   6.75        81.0
2        12   6.75        81.0
3        48   2.10       100.8
4        24   1.25        30.0

Negative TotalPrice: 0
Zero TotalPrice: 0


In [50]:
# Validate the cleaned dataset before temporal analysis

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Missing values:", df.isna().sum().sum())
print("Negative quantities:", (df["Quantity"] < 0).sum())
print("Zero quantities:", (df["Quantity"] == 0).sum())
print("Negative prices:", (df["Price"] < 0).sum())
print("Zero prices:", (df["Price"] == 0).sum())
print("Exact duplicates:", df.duplicated().sum())

Rows: 776844
Columns: 9
Missing values: 0
Negative quantities: 0
Zero quantities: 0
Negative prices: 0
Zero prices: 0
Exact duplicates: 0


In [53]:
# Aggregate transactions into customer-month behavioral features

customer_month = (
    df.groupby(["Customer ID", "Month"])
    .agg(
        transaction_count=("Invoice", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_spending=("TotalPrice", "sum"),
        average_transaction_value=("TotalPrice", "mean"),
        unique_products=("StockCode", "nunique")
    )
    .reset_index()
)

customer_month.head()

,Customer ID,Month,transaction_count,total_quantity,total_spending,average_transaction_value,unique_products
0,12346.0,2010-03,1,5,27.05,5.410000,5
1,12346.0,2010-06,1,19,142.31,7.490000,19
2,12346.0,2011-01,1,74215,77183.60,77183.600000,1
3,12347.0,2010-10,1,509,611.53,15.288250,40
4,12347.0,2010-12,1,319,711.79,22.960968,31


In [54]:
# Validate the customer-month dataset

print("Customer-month rows:", len(customer_month))
print("Unique customers:", customer_month["Customer ID"].nunique())
print("Unique months:", customer_month["Month"].nunique())

print("\nMissing values:")
print(customer_month.isna().sum())

print("\nDuplicate customer-month pairs:",
      customer_month.duplicated(
          subset=["Customer ID", "Month"]
      ).sum())

Customer-month rows: 25504
Unique customers: 5853
Unique months: 25

Missing values:
Customer ID                  0
Month                        0
transaction_count            0
total_quantity               0
total_spending               0
average_transaction_value    0
unique_products              0
dtype: int64

Duplicate customer-month pairs: 0


In [55]:
# Inspect the distribution of monthly behavioral features

behavior_features = [
    "transaction_count",
    "total_quantity",
    "total_spending",
    "average_transaction_value",
    "unique_products"
]

customer_month[behavior_features].describe()

,transaction_count,total_quantity,total_spending,average_transaction_value,unique_products
count,25504.000000,25504.000000,25504.000000,25504.000000,25504.000000
mean,1.435069,411.661073,669.743128,47.782461,28.182011
std,1.288341,1681.932659,2132.380047,1168.380057,32.984161
min,1.000000,1.000000,0.850000,0.850000,1.000000
25%,1.000000,102.000000,206.622500,12.593750,10.000000
50%,1.000000,196.000000,344.245000,18.016696,19.000000
75%,1.000000,366.000000,612.425000,28.126282,35.000000
max,44.000000,93230.000000,168469.600000,168469.600000,885.000000


In [56]:
# Calculate how many months each customer was active

customer_activity = (
    customer_month.groupby("Customer ID")["Month"]
    .nunique()
)

print(customer_activity.describe())

print("\nCustomers by number of active months:")
print(customer_activity.value_counts().sort_index())

count    5853.000000
mean        4.357424
std         4.492233
min         1.000000
25%         1.000000
50%         3.000000
75%         6.000000
max        25.000000
Name: Month, dtype: float64

Customers by number of active months:
Month
1     1759
2     1073
3      659
4      495
5      384
6      281
7      195
8      185
9      149
10     115
11      89
12      75
13      53
14      60
15      38
16      48
17      32
18      28
19      33
20      27
21      16
22       9
23      15
24      15
25      20
Name: count, dtype: int64


In [57]:
# Sort customer-month observations chronologically for each customer

customer_month_sorted = customer_month.sort_values(
    ["Customer ID", "Month"]
).copy()

print(customer_month_sorted.head(10))

   Customer ID    Month  transaction_count  total_quantity  total_spending  \
0      12346.0  2010-03                  1               5           27.05   
1      12346.0  2010-06                  1              19          142.31   
2      12346.0  2011-01                  1           74215        77183.60   
3      12347.0  2010-10                  1             509          611.53   
4      12347.0  2010-12                  1             319          711.79   
5      12347.0  2011-01                  1             315          475.39   
6      12347.0  2011-04                  1             483          636.25   
7      12347.0  2011-06                  1             196          382.52   
8      12347.0  2011-08                  1             277          584.91   
9      12347.0  2011-10                  1             676         1294.32   

   average_transaction_value  unique_products  
0                   5.410000                5  
1                   7.490000               19

In [58]:
# Identify the previous active month for each customer

customer_month_sorted["previous_month"] = (
    customer_month_sorted
    .groupby("Customer ID")["Month"]
    .shift(1)
)

print(customer_month_sorted[
    ["Customer ID", "Month", "previous_month"]
].head(10))

   Customer ID    Month previous_month
0      12346.0  2010-03            NaT
1      12346.0  2010-06        2010-03
2      12346.0  2011-01        2010-06
3      12347.0  2010-10            NaT
4      12347.0  2010-12        2010-10
5      12347.0  2011-01        2010-12
6      12347.0  2011-04        2011-01
7      12347.0  2011-06        2011-04
8      12347.0  2011-08        2011-06
9      12347.0  2011-10        2011-08


In [59]:
# Calculate the number of months since the previous active month

customer_month_sorted["months_since_previous"] = (
    customer_month_sorted["Month"]
    - customer_month_sorted["previous_month"]
).apply(
    lambda x: x.n if pd.notna(x) else pd.NA
)

print(customer_month_sorted[
    ["Customer ID", "Month", "previous_month", "months_since_previous"]
].head(10))

   Customer ID    Month previous_month months_since_previous
0      12346.0  2010-03            NaT                  <NA>
1      12346.0  2010-06        2010-03                     3
2      12346.0  2011-01        2010-06                     7
3      12347.0  2010-10            NaT                  <NA>
4      12347.0  2010-12        2010-10                     2
5      12347.0  2011-01        2010-12                     1
6      12347.0  2011-04        2011-01                     3
7      12347.0  2011-06        2011-04                     2
8      12347.0  2011-08        2011-06                     2
9      12347.0  2011-10        2011-08                     2


In [60]:
# Inspect the distribution of gaps between active months

gap_data = customer_month_sorted[
    customer_month_sorted["months_since_previous"].notna()
]["months_since_previous"]

print("Customer-month observations with a previous month:", len(gap_data))

print("\nGap distribution:")
print(gap_data.describe())

print("\nGap frequencies:")
print(gap_data.value_counts().sort_index())

Customer-month observations with a previous month: 19651

Gap distribution:
count     19651
unique       23
top           1
freq       9330
Name: months_since_previous, dtype: int64

Gap frequencies:
months_since_previous
1     9330
2     3983
3     2059
4     1237
5      824
6      526
7      426
8      266
9      181
10     187
11     197
12     178
13     105
14      41
15      22
16      15
17      26
18      14
19      12
20      10
21       5
22       5
23       2
Name: count, dtype: int64


In [61]:
# Check how many observations have consecutive active months

consecutive_months = customer_month_sorted[
    customer_month_sorted["months_since_previous"] == 1
].copy()

print("Consecutive customer-month observations:", len(consecutive_months))

print(
    "Unique customers with consecutive months:",
    consecutive_months["Customer ID"].nunique()
)

Consecutive customer-month observations: 9330
Unique customers with consecutive months: 2425


In [62]:
# Create previous-month behavioral features

for feature in behavior_features:
    customer_month_sorted[f"previous_{feature}"] = (
        customer_month_sorted
        .groupby("Customer ID")[feature]
        .shift(1)
    )

print(customer_month_sorted[
    ["Customer ID", "Month"] +
    [f"previous_{feature}" for feature in behavior_features]
].head(10))

   Customer ID    Month  previous_transaction_count  previous_total_quantity  \
0      12346.0  2010-03                         NaN                      NaN   
1      12346.0  2010-06                         1.0                      5.0   
2      12346.0  2011-01                         1.0                     19.0   
3      12347.0  2010-10                         NaN                      NaN   
4      12347.0  2010-12                         1.0                    509.0   
5      12347.0  2011-01                         1.0                    319.0   
6      12347.0  2011-04                         1.0                    315.0   
7      12347.0  2011-06                         1.0                    483.0   
8      12347.0  2011-08                         1.0                    196.0   
9      12347.0  2011-10                         1.0                    277.0   

   previous_total_spending  previous_average_transaction_value  \
0                      NaN                           

In [63]:
# Calculate absolute changes in customer behavior

change_features = behavior_features.copy()

for feature in change_features:
    customer_month_sorted[f"change_{feature}"] = (
        customer_month_sorted[feature]
        - customer_month_sorted[f"previous_{feature}"]
    )

customer_month_sorted[
    ["Customer ID", "Month"] +
    [f"change_{feature}" for feature in change_features]
].head(10)

,Customer ID,Month,change_transaction_count,change_total_quantity,change_total_spending,change_average_transaction_value,change_unique_products
0,12346.0,2010-03,NaN,NaN,NaN,NaN,NaN
1,12346.0,2010-06,0.0,14.0,115.26,2.080000,14.0
2,12346.0,2011-01,0.0,74196.0,77041.29,77176.110000,-18.0
3,12347.0,2010-10,NaN,NaN,NaN,NaN,NaN
4,12347.0,2010-12,0.0,-190.0,100.26,7.672718,-9.0
5,12347.0,2011-01,0.0,-4.0,-236.40,-6.568209,-2.0
6,12347.0,2011-04,0.0,168.0,160.86,10.117658,-5.0
7,12347.0,2011-06,0.0,-287.0,-253.73,-5.259306,-6.0
8,12347.0,2011-08,0.0,81.0,202.39,5.335707,4.0
9,12347.0,2011-10,0.0,399.0,709.41,0.951905,25.0


In [64]:
# Calculate percentage changes in customer behavior

for feature in change_features:
    previous_values = customer_month_sorted[f"previous_{feature}"]
    
    customer_month_sorted[f"pct_change_{feature}"] = np.where(
        previous_values != 0,
        (
            customer_month_sorted[feature] - previous_values
        ) / previous_values * 100,
        np.nan
    )

percentage_change_features = [
    f"pct_change_{feature}"
    for feature in change_features
]

customer_month_sorted[
    ["Customer ID", "Month"] + percentage_change_features
].head(10)

,Customer ID,Month,pct_change_transaction_count,pct_change_total_quantity,pct_change_total_spending,pct_change_average_transaction_value,pct_change_unique_products
0,12346.0,2010-03,NaN,NaN,NaN,NaN,NaN
1,12346.0,2010-06,0.0,280.000000,426.099815,3.844732e+01,280.000000
2,12346.0,2011-01,0.0,390505.263158,54136.244818,1.030389e+06,-94.736842
3,12347.0,2010-10,NaN,NaN,NaN,NaN,NaN
4,12347.0,2010-12,0.0,-37.328094,16.394944,5.018702e+01,-22.500000
5,12347.0,2011-01,0.0,-1.253918,-33.212043,-2.860598e+01,-6.451613
6,12347.0,2011-04,0.0,53.333333,33.837481,6.172029e+01,-17.241379
7,12347.0,2011-06,0.0,-59.420290,-39.878978,-1.983864e+01,-25.000000
8,12347.0,2011-08,0.0,41.326531,52.909652,2.510790e+01,22.222222
9,12347.0,2011-10,0.0,144.043321,121.285326,3.580365e+00,113.636364


In [65]:
# Inspect the distribution of percentage changes

customer_month_sorted[
    percentage_change_features
].describe().T

,count,mean,std,min,25%,50%,75%,max
pct_change_transaction_count,19651.0,12.093704,62.689142,-90.000000,0.000000,0.000000,0.000000,1.300000e+03
pct_change_total_quantity,19651.0,371.370971,29177.826768,-99.951776,-43.581706,-0.689655,68.085106,4.049650e+06
pct_change_total_spending,19651.0,390.196239,41553.134675,-99.956424,-37.863187,-1.143285,52.981336,5.809197e+06
pct_change_average_transaction_value,19651.0,676.162480,83207.495527,-99.845825,-20.000000,2.090127,30.293022,1.161849e+07
pct_change_unique_products,19651.0,51.434707,303.276621,-99.462366,-40.740741,0.000000,53.333333,1.148000e+04


In [66]:
# Identify extreme percentage changes

extreme_threshold = 500

extreme_counts = (
    customer_month_sorted[percentage_change_features]
    .abs()
    .gt(extreme_threshold)
    .sum()
)

print("Observations with absolute percentage change > 500%:")
print(extreme_counts)

print("\nTotal affected observations:")
print(
    customer_month_sorted[percentage_change_features]
    .abs()
    .gt(extreme_threshold)
    .any(axis=1)
    .sum()
)

Observations with absolute percentage change > 500%:
pct_change_transaction_count             11
pct_change_total_quantity               613
pct_change_total_spending               353
pct_change_average_transaction_value    168
pct_change_unique_products              502
dtype: int64

Total affected observations:
1028


In [67]:
# Inspect extreme percentage changes using percentiles

percentile_summary = (
    customer_month_sorted[percentage_change_features]
    .quantile([0.01, 0.99])
    .T
)

percentile_summary.columns = ["P1", "P99"]

percentile_summary

,P1,P99
pct_change_transaction_count,-68.333333,200.000000
pct_change_total_quantity,-94.352226,1302.173913
pct_change_total_spending,-89.822732,766.053656
pct_change_average_transaction_value,-82.643222,455.959350
pct_change_unique_products,-92.754167,1000.000000


In [68]:
# Inspect extreme percentage changes using percentiles

percentile_summary = (
    customer_month_sorted[percentage_change_features]
    .quantile([0.01, 0.99])
    .T
)

percentile_summary.columns = ["P1", "P99"]

percentile_summary

,P1,P99
pct_change_transaction_count,-68.333333,200.000000
pct_change_total_quantity,-94.352226,1302.173913
pct_change_total_spending,-89.822732,766.053656
pct_change_average_transaction_value,-82.643222,455.959350
pct_change_unique_products,-92.754167,1000.000000


In [69]:
# Keep observations with a previous active month

behavior_change_df = customer_month_sorted[
    customer_month_sorted["previous_month"].notna()
].copy()

print("Observations with previous month:", len(behavior_change_df))
print(
    "Unique customers with previous month:",
    behavior_change_df["Customer ID"].nunique()
)

Observations with previous month: 19651
Unique customers with previous month: 4094


In [70]:
# Analyze large behavioral changes across customers

change_summary = (
    behavior_change_df[percentage_change_features]
    .abs()
    .gt(50)
    .sum()
)

print("Observations with absolute percentage change > 50%:")
print(change_summary)

print("\nTotal observations with at least one feature changing > 50%:")

affected = (
    behavior_change_df[percentage_change_features]
    .abs()
    .gt(50)
    .any(axis=1)
)

print(affected.sum())
print("Percentage of observations:", round(affected.mean() * 100, 2), "%")

Observations with absolute percentage change > 50%:
pct_change_transaction_count            3809
pct_change_total_quantity               9910
pct_change_total_spending               8483
pct_change_average_transaction_value    4772
pct_change_unique_products              8563
dtype: int64

Total observations with at least one feature changing > 50%:
14127
Percentage of observations: 71.89 %


In [71]:
# Compare candidate thresholds for behavioral change

thresholds = [50, 75, 100, 200]

for threshold in thresholds:
    affected = (
        behavior_change_df[percentage_change_features]
        .abs()
        .gt(threshold)
        .any(axis=1)
    )

    print(
        f"Threshold {threshold}%: "
        f"{affected.sum()} observations "
        f"({affected.mean() * 100:.2f}%)"
    )

Threshold 50%: 14127 observations (71.89%)
Threshold 75%: 9863 observations (50.19%)
Threshold 100%: 6314 observations (32.13%)
Threshold 200%: 3274 observations (16.66%)


In [72]:
# Count how many behavioral features changed by more than 100%

threshold = 100

behavior_change_df["large_change_count"] = (
    behavior_change_df[percentage_change_features]
    .abs()
    .gt(threshold)
    .sum(axis=1)
)

print("Distribution of large behavioral changes:")
print(
    behavior_change_df["large_change_count"]
    .value_counts()
    .sort_index()
)

print("\nPercentage distribution:")
print(
    behavior_change_df["large_change_count"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Distribution of large behavioral changes:
large_change_count
0    13337
1     3001
2     1282
3     1596
4      416
5       19
Name: count, dtype: int64

Percentage distribution:
large_change_count
0    67.87
1    15.27
2     6.52
3     8.12
4     2.12
5     0.10
Name: proportion, dtype: float64


In [73]:
# Inspect behavior-shift candidates across customers

behavior_change_df["behavior_shift_candidate"] = (
    behavior_change_df["large_change_count"] >= 2
)

shift_summary = (
    behavior_change_df
    .groupby("Customer ID")["behavior_shift_candidate"]
    .agg(
        shift_observations="sum",
        total_observations="count"
    )
)

shift_summary["shift_rate"] = (
    shift_summary["shift_observations"]
    / shift_summary["total_observations"]
)

print(
    "Customers with at least one shift candidate:",
    (shift_summary["shift_observations"] > 0).sum()
)

print(
    "\nShift observations:",
    behavior_change_df["behavior_shift_candidate"].sum()
)

print(
    "\nOverall shift rate:",
    round(
        behavior_change_df["behavior_shift_candidate"].mean() * 100,
        2
    ),
    "%"
)

Customers with at least one shift candidate: 1941

Shift observations: 3313

Overall shift rate: 16.86 %


In [74]:
# Inspect behavior-shift candidates over time

monthly_shift_summary = (
    behavior_change_df
    .groupby("Month")["behavior_shift_candidate"]
    .agg(
        shift_observations="sum",
        total_observations="count"
    )
)

monthly_shift_summary["shift_rate"] = (
    monthly_shift_summary["shift_observations"]
    / monthly_shift_summary["total_observations"]
    * 100
)

monthly_shift_summary

,shift_observations,total_observations,shift_rate
Month,,,
2010-01,40,333,12.012012
2010-02,59,396,14.898990
2010-03,89,610,14.590164
2010-04,100,645,15.503876
2010-05,105,710,14.788732
2010-06,109,767,14.211213
2010-07,110,739,14.884980
2010-08,116,747,15.528782
2010-09,196,897,21.850613


In [75]:
# Summary statistics for monthly shift rates

print("Minimum monthly shift rate:",
      round(monthly_shift_summary["shift_rate"].min(), 2), "%")

print("Maximum monthly shift rate:",
      round(monthly_shift_summary["shift_rate"].max(), 2), "%")

print("Average monthly shift rate:",
      round(monthly_shift_summary["shift_rate"].mean(), 2), "%")

Minimum monthly shift rate: 7.17 %
Maximum monthly shift rate: 23.99 %
Average monthly shift rate: 15.99 %


In [76]:
# Inspect previous-month baseline values for shift candidates

shift_candidates = behavior_change_df[
    behavior_change_df["behavior_shift_candidate"]
].copy()

baseline_features = [
    "previous_transaction_count",
    "previous_total_quantity",
    "previous_total_spending",
    "previous_average_transaction_value",
    "previous_unique_products"
]

shift_candidates[baseline_features].describe().T

,count,mean,std,min,25%,50%,75%,max
previous_transaction_count,3313.0,1.210987,0.795807,1.00,1.000,1.000000,1.000000,15.00
previous_total_quantity,3313.0,229.485361,665.556562,1.00,54.000,108.000000,218.000000,17520.00
previous_total_spending,3313.0,395.915778,924.815328,0.85,127.500,227.930000,368.150000,18165.74
previous_average_transaction_value,3313.0,41.781964,109.076943,0.85,12.575,19.185217,34.582222,3707.40
previous_unique_products,3313.0,16.711742,21.740813,1.00,5.000,12.000000,20.000000,579.00


In [77]:
# Inspect baseline spending and product activity for shift candidates

print("Previous spending percentiles:")
print(
    shift_candidates["previous_total_spending"]
    .quantile([0.10, 0.25, 0.50, 0.75, 0.90])
)

print("\nPrevious unique products percentiles:")
print(
    shift_candidates["previous_unique_products"]
    .quantile([0.10, 0.25, 0.50, 0.75, 0.90])
)

print("\nPrevious transaction count distribution:")
print(
    shift_candidates["previous_transaction_count"]
    .value_counts()
    .sort_index()
    .head(15)
)

Previous spending percentiles:
0.10     77.510
0.25    127.500
0.50    227.930
0.75    368.150
0.90    650.676
Name: previous_total_spending, dtype: float64

Previous unique products percentiles:
0.10     2.0
0.25     5.0
0.50    12.0
0.75    20.0
0.90    34.0
Name: previous_unique_products, dtype: float64

Previous transaction count distribution:
previous_transaction_count
1.0     2901
2.0      294
3.0       51
4.0       33
5.0       13
6.0        7
7.0        4
8.0        4
9.0        1
10.0       1
12.0       2
13.0       1
15.0       1
Name: count, dtype: int64


In [78]:
# Define core behavioral dimensions for behavior shift detection

core_behavior_features = [
    "total_quantity",
    "total_spending",
    "average_transaction_value",
    "unique_products"
]

core_percentage_change_features = [
    f"pct_change_{feature}"
    for feature in core_behavior_features
]

print("Core behavioral dimensions:")
print(core_behavior_features)

Core behavioral dimensions:
['total_quantity', 'total_spending', 'average_transaction_value', 'unique_products']


In [79]:
# Count large changes in core behavioral dimensions

core_threshold = 100

behavior_change_df["core_large_change_count"] = (
    behavior_change_df[core_percentage_change_features]
    .abs()
    .gt(core_threshold)
    .sum(axis=1)
)

print("Distribution of large changes in core behavioral dimensions:")
print(
    behavior_change_df["core_large_change_count"]
    .value_counts()
    .sort_index()
)

print("\nPercentage distribution:")
print(
    behavior_change_df["core_large_change_count"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Distribution of large changes in core behavioral dimensions:
core_large_change_count
0    13513
1     2901
2     1340
3     1790
4      107
Name: count, dtype: int64

Percentage distribution:
core_large_change_count
0    68.76
1    14.76
2     6.82
3     9.11
4     0.54
Name: proportion, dtype: float64


In [80]:
# Define the core behavior shift candidate

behavior_change_df["core_behavior_shift_candidate"] = (
    behavior_change_df["core_large_change_count"] >= 2
)

print(
    "Core behavior shift observations:",
    behavior_change_df["core_behavior_shift_candidate"].sum()
)

print(
    "Core behavior shift rate:",
    round(
        behavior_change_df["core_behavior_shift_candidate"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Customers with at least one core behavior shift:",
    behavior_change_df.loc[
        behavior_change_df["core_behavior_shift_candidate"],
        "Customer ID"
    ].nunique()
)

Core behavior shift observations: 3237
Core behavior shift rate: 16.47 %
Customers with at least one core behavior shift: 1923


In [81]:
# Create the final behavior shift target

behavior_change_df["behavior_shift"] = (
    behavior_change_df["core_behavior_shift_candidate"]
    .astype(int)
)

print("Target distribution:")
print(
    behavior_change_df["behavior_shift"]
    .value_counts()
    .sort_index()
)

print("\nTarget percentage:")
print(
    behavior_change_df["behavior_shift"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Target distribution:
behavior_shift
0    16414
1     3237
Name: count, dtype: int64

Target percentage:
behavior_shift
0    83.53
1    16.47
Name: proportion, dtype: float64


In [82]:
# Finalize the behavior shift target

behavior_change_df["behavior_shift"] = (
    behavior_change_df["core_large_change_count"] >= 2
).astype(int)

print("Target distribution:")
print(behavior_change_df["behavior_shift"].value_counts().sort_index())

print("\nTarget percentage:")
print(
    behavior_change_df["behavior_shift"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Target distribution:
behavior_shift
0    16414
1     3237
Name: count, dtype: int64

Target percentage:
behavior_shift
0    83.53
1    16.47
Name: proportion, dtype: float64


In [83]:
# Inspect available features before leakage analysis

print("Available columns:")
for column in behavior_change_df.columns:
    print(column)

Available columns:
Customer ID
Month
transaction_count
total_quantity
total_spending
average_transaction_value
unique_products
previous_month
months_since_previous
previous_transaction_count
previous_total_quantity
previous_total_spending
previous_average_transaction_value
previous_unique_products
change_transaction_count
change_total_quantity
change_total_spending
change_average_transaction_value
change_unique_products
pct_change_transaction_count
pct_change_total_quantity
pct_change_total_spending
pct_change_average_transaction_value
pct_change_unique_products
large_change_count
behavior_shift_candidate
core_large_change_count
core_behavior_shift_candidate
behavior_shift


In [84]:
# Define leakage-safe feature groups

static_features = [
    "Customer ID",
    "Country"
]

previous_behavior_features = [
    "previous_transaction_count",
    "previous_total_quantity",
    "previous_total_spending",
    "previous_average_transaction_value",
    "previous_unique_products",
    "months_since_previous"
]

target_column = "behavior_shift"

print("Static features:")
print(static_features)

print("\nPrevious behavioral features:")
print(previous_behavior_features)

print("\nTarget:")
print(target_column)

Static features:
['Customer ID', 'Country']

Previous behavioral features:
['previous_transaction_count', 'previous_total_quantity', 'previous_total_spending', 'previous_average_transaction_value', 'previous_unique_products', 'months_since_previous']

Target:
behavior_shift


In [85]:
# Inspect the temporal distribution before splitting

monthly_observations = (
    behavior_change_df
    .groupby("Month")
    .size()
    .reset_index(name="observations")
)

print(monthly_observations)

print("\nNumber of months:", len(monthly_observations))

      Month  observations
0   2010-01           333
1   2010-02           396
2   2010-03           610
3   2010-04           645
4   2010-05           710
5   2010-06           767
6   2010-07           739
7   2010-08           747
8   2010-09           897
9   2010-10          1117
10  2010-11          1280
11  2010-12           808
12  2011-01           667
13  2011-02           632
14  2011-03           794
15  2011-04           747
16  2011-05           943
17  2011-06           882
18  2011-07           845
19  2011-08           826
20  2011-09          1071
21  2011-10          1140
22  2011-11          1469
23  2011-12           586

Number of months: 24


In [86]:
# Create a leakage-aware time-based train/validation/test split

train_end = "2011-05"
validation_end = "2011-08"

train_df = behavior_change_df[
    behavior_change_df["Month"] <= train_end
].copy()

validation_df = behavior_change_df[
    (behavior_change_df["Month"] > train_end) &
    (behavior_change_df["Month"] <= validation_end)
].copy()

test_df = behavior_change_df[
    behavior_change_df["Month"] > validation_end
].copy()

print("Train period:", train_df["Month"].min(), "to", train_df["Month"].max())
print("Validation period:", validation_df["Month"].min(), "to", validation_df["Month"].max())
print("Test period:", test_df["Month"].min(), "to", test_df["Month"].max())

print("\nRows:")
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

Train period: 2010-01 to 2011-05
Validation period: 2011-06 to 2011-08
Test period: 2011-09 to 2011-12

Rows:
Train: 12832
Validation: 2553
Test: 4266


In [87]:
# Check target distribution across train, validation, and test sets

for name, dataset in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(f"\n{name} target distribution:")
    print(dataset["behavior_shift"].value_counts())
    print(
        dataset["behavior_shift"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )


Train target distribution:
behavior_shift
0    10745
1     2087
Name: count, dtype: int64
behavior_shift
0    83.74
1    16.26
Name: proportion, dtype: float64

Validation target distribution:
behavior_shift
0    2198
1     355
Name: count, dtype: int64
behavior_shift
0    86.09
1    13.91
Name: proportion, dtype: float64

Test target distribution:
behavior_shift
0    3471
1     795
Name: count, dtype: int64
behavior_shift
0    81.36
1    18.64
Name: proportion, dtype: float64


In [88]:
# Create historical customer activity features using information available before the current month

behavior_change_df = behavior_change_df.sort_values(
    ["Customer ID", "Month"]
).copy()

behavior_change_df["historical_active_months"] = (
    behavior_change_df
    .groupby("Customer ID")
    .cumcount()
)

behavior_change_df["historical_transactions"] = (
    behavior_change_df
    .groupby("Customer ID")["previous_transaction_count"]
    .cumsum()
)

behavior_change_df["historical_spending"] = (
    behavior_change_df
    .groupby("Customer ID")["previous_total_spending"]
    .cumsum()
)

behavior_change_df[
    [
        "Customer ID",
        "Month",
        "historical_active_months",
        "historical_transactions",
        "historical_spending"
    ]
].head(10)

,Customer ID,Month,historical_active_months,historical_transactions,historical_spending
1,12346.0,2010-06,0,1.0,27.05
2,12346.0,2011-01,1,2.0,169.36
4,12347.0,2010-12,0,1.0,611.53
5,12347.0,2011-01,1,2.0,1323.32
6,12347.0,2011-04,2,3.0,1798.71
7,12347.0,2011-06,3,4.0,2434.96
8,12347.0,2011-08,4,5.0,2817.48
9,12347.0,2011-10,5,6.0,3402.39
10,12347.0,2011-12,6,7.0,4696.71
12,12348.0,2010-12,0,1.0,221.16


In [89]:
# Recreate the time-based splits with the historical features

train_df = behavior_change_df[
    behavior_change_df["Month"] <= "2011-05"
].copy()

validation_df = behavior_change_df[
    (behavior_change_df["Month"] > "2011-05") &
    (behavior_change_df["Month"] <= "2011-08")
].copy()

test_df = behavior_change_df[
    behavior_change_df["Month"] > "2011-08"
].copy()

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (12832, 32)
Validation: (2553, 32)
Test: (4266, 32)


In [90]:
# Recreate time-based splits after adding historical features

train_df = behavior_change_df[
    behavior_change_df["Month"] <= train_end
].copy()

validation_df = behavior_change_df[
    (behavior_change_df["Month"] > train_end) &
    (behavior_change_df["Month"] <= validation_end)
].copy()

test_df = behavior_change_df[
    behavior_change_df["Month"] > validation_end
].copy()

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (12832, 32)
Validation: (2553, 32)
Test: (4266, 32)


In [93]:
# Define baseline and behavior-aware feature sets

baseline_features = [
    "historical_active_months",
    "historical_transactions",
    "historical_spending"
]

behavior_aware_features = baseline_features + [
    "previous_transaction_count",
    "previous_total_quantity",
    "previous_total_spending",
    "previous_average_transaction_value",
    "previous_unique_products",
    "months_since_previous"
]

print("Baseline features:")
print(baseline_features)

print("\nBehavior-aware features:")
print(behavior_aware_features)

Baseline features:
['historical_active_months', 'historical_transactions', 'historical_spending']

Behavior-aware features:
['historical_active_months', 'historical_transactions', 'historical_spending', 'previous_transaction_count', 'previous_total_quantity', 'previous_total_spending', 'previous_average_transaction_value', 'previous_unique_products', 'months_since_previous']


In [94]:
# Check missing values and data types for model features

for name, features in [
    ("Baseline", baseline_features),
    ("Behavior-aware", behavior_aware_features)
]:
    print(f"\n{name} features:")
    print("\nMissing values:")
    print(train_df[features].isna().sum())

    print("\nData types:")
    print(train_df[features].dtypes)


Baseline features:

Missing values:
historical_active_months    0
historical_transactions     0
historical_spending         0
dtype: int64

Data types:
historical_active_months      int64
historical_transactions     float64
historical_spending         float64
dtype: object

Behavior-aware features:

Missing values:
historical_active_months              0
historical_transactions               0
historical_spending                   0
previous_transaction_count            0
previous_total_quantity               0
previous_total_spending               0
previous_average_transaction_value    0
previous_unique_products              0
months_since_previous                 0
dtype: int64

Data types:
historical_active_months                int64
historical_transactions               float64
historical_spending                   float64
previous_transaction_count            float64
previous_total_quantity               float64
previous_total_spending               float64
previous_average_tra

In [95]:
# Convert months_since_previous to numeric

for dataset in [train_df, validation_df, test_df]:
    dataset["months_since_previous"] = pd.to_numeric(
        dataset["months_since_previous"],
        errors="coerce"
    )

print("months_since_previous data type:")
print(train_df["months_since_previous"].dtype)

print("\nMissing values after conversion:")
print(
    train_df[
        behavior_aware_features
    ].isna().sum()
)

months_since_previous data type:
int64

Missing values after conversion:
historical_active_months              0
historical_transactions               0
historical_spending                   0
previous_transaction_count            0
previous_total_quantity               0
previous_total_spending               0
previous_average_transaction_value    0
previous_unique_products              0
months_since_previous                 0
dtype: int64


In [96]:
# Inspect the distribution of model features

print("Baseline feature distribution:")
print(train_df[baseline_features].describe())

print("\nBehavior-aware feature distribution:")
print(train_df[behavior_aware_features].describe())

Baseline feature distribution:
       historical_active_months  historical_transactions  historical_spending
count              12832.000000             12832.000000         12832.000000
mean                   3.081047                 7.415680          4079.438097
std                    3.319334                12.584977         15169.302436
min                    0.000000                 1.000000             1.300000
25%                    0.000000                 2.000000           523.650000
50%                    2.000000                 4.000000          1280.140000
75%                    5.000000                 8.000000          3048.135000
max                   16.000000               220.000000        367806.110000

Behavior-aware feature distribution:
       historical_active_months  historical_transactions  historical_spending  \
count              12832.000000             12832.000000         12832.000000   
mean                   3.081047                 7.415680          4

In [97]:
# Check feature skewness before preprocessing

print("Baseline feature skewness:")
print(
    train_df[baseline_features]
    .skew()
    .sort_values(ascending=False)
)

print("\nBehavior-aware feature skewness:")
print(
    train_df[behavior_aware_features]
    .skew()
    .sort_values(ascending=False)
)

Baseline feature skewness:
historical_spending         13.409214
historical_transactions      6.374778
historical_active_months     1.352343
dtype: float64

Behavior-aware feature skewness:
previous_total_quantity               30.098636
previous_average_transaction_value    17.449372
historical_spending                   13.409214
previous_total_spending               13.094023
previous_transaction_count             8.188063
historical_transactions                6.374778
previous_unique_products               4.436359
months_since_previous                  2.297009
historical_active_months               1.352343
dtype: float64


In [98]:
# Identify highly right-skewed features for log transformation

skew_threshold = 1.0

baseline_skewed_features = [
    feature
    for feature in baseline_features
    if train_df[feature].skew() > skew_threshold
]

behavior_aware_skewed_features = [
    feature
    for feature in behavior_aware_features
    if train_df[feature].skew() > skew_threshold
]

print("Baseline features requiring log transformation:")
print(baseline_skewed_features)

print("\nBehavior-aware features requiring log transformation:")
print(behavior_aware_skewed_features)

Baseline features requiring log transformation:
['historical_active_months', 'historical_transactions', 'historical_spending']

Behavior-aware features requiring log transformation:
['historical_active_months', 'historical_transactions', 'historical_spending', 'previous_transaction_count', 'previous_total_quantity', 'previous_total_spending', 'previous_average_transaction_value', 'previous_unique_products', 'months_since_previous']


In [99]:
# Define features that require log transformation

baseline_log_features = [
    "historical_transactions",
    "historical_spending"
]

behavior_aware_log_features = [
    "historical_transactions",
    "historical_spending",
    "previous_transaction_count",
    "previous_total_quantity",
    "previous_total_spending",
    "previous_average_transaction_value",
    "previous_unique_products"
]

print("Baseline log-transformed features:")
print(baseline_log_features)

print("\nBehavior-aware log-transformed features:")
print(behavior_aware_log_features)

Baseline log-transformed features:
['historical_transactions', 'historical_spending']

Behavior-aware log-transformed features:
['historical_transactions', 'historical_spending', 'previous_transaction_count', 'previous_total_quantity', 'previous_total_spending', 'previous_average_transaction_value', 'previous_unique_products']
